# snow Tutorial 1 - Introduction

snow simulates the propagation of ultrashort pulses through quadratic ($\chi^{(2)}$) nonlinear media: dispersion-engineered thin-film lithium niobate waveguides, bulk crystals, and optical parametric oscillators built from them. It solves a single envelope equation that contains every three-wave process at once (second-harmonic generation, sum- and difference-frequency generation, parametric amplification and generation) over a bandwidth that can span more than an octave.

## What snow can do

- **$\chi^{(2)}$ pulse propagation** with the nonlinear envelope equation (NEE), including cascaded processes and vacuum-noise seeding for parametric generation.
- **Waveguide dispersion** from three interchangeable index models: `RidgeIndex` (effective index method for a ridge geometry), `BehavioralIndex` (a waveguide defined by its zero-GVD wavelength, GVM, ...) and `TabulatedIndex` (your own mode-solver data).
- **A material library** of Sellmeier models (MgO:LN, LN, LT, SiO₂, ...) with temperature dependence: `snow.materials.available()` lists them.
- **Quasi-phase matching**, either through a grating wavevector (fast, for uniform gratings) or an arbitrary poling function d(z) (chirped or apodized gratings).
- **Bulk crystals** (`snow.crystals`) and **synchronously pumped OPOs** (`snow.opos`), which run round trip after round trip through either medium.
- **Lumerical MODE helpers** (`snow.lumerical`) to draw the same ridge in a full-vectorial mode solver.

Planned: $\chi^{(3)}$, Raman scattering, Gaussian beams for bulk simulations, full EM mode simulations, multimode and backward propagation.

## The model

snow describes everything in the waveguide by one complex envelope $a(t, z)$ around a reference frequency $\omega_0$, scaled so that $|a|^2$ is the instantaneous power in W. Its spectrum $A(\Omega, z)$, with $\Omega = \omega - \omega_0$, obeys

$$
\frac{\partial A}{\partial z} = -i\,D(\Omega)\,A \;-\; i\,k(z,\Omega)\;\mathcal{F}\!\left\{ a^2 e^{i\phi} + 2|a|^2 e^{-i\phi} \right\},
$$

with

$$
D(\Omega) = \beta(\omega_0+\Omega) - \beta_0 - \frac{\Omega}{v_\mathrm{ref}} - \frac{i\alpha}{2},
\qquad
\phi(t,z) = \omega_0 t - \left(\beta_0 - \frac{\omega_0}{v_\mathrm{ref}} + K_g\right) z,
\qquad
k(z,\omega) = \frac{d(z)\,X_0\,\omega}{4N}.
$$

- $D$ is the linear part: the full dispersion $\beta(\omega)$ of the mode, written in a frame that moves at velocity $v_\mathrm{ref}$, plus the loss $\alpha$.
- $a^2 e^{i\phi}$ generates sum frequencies (SHG, SFG). $2|a|^2e^{-i\phi}$ generates difference frequencies (DFG, OPA). Because a single envelope carries all the waves, nothing has to be split into "pump", "signal" and "idler", and overlapping spectra are handled correctly.
- $d(z) = \pm1$ is the poling, $X_0$ the effective nonlinearity and $N$ a mode normalization; $K_g$ is an optional grating wavevector for quasi-phase matching.

The solver works in the interaction picture (the linear part is exact) and integrates the nonlinear term with an adaptive Runge-Kutta method. The full derivation of the conventions is in [docs/physics.md](../docs/physics.md).

**Conventions.** SI units throughout. Fields vary as $e^{i(\omega t - \beta z)}$. Spectra are stored in FFT order (zero frequency first), as Tutorial 2 explains.

## Workflow

A simulation has four steps, and each has its own tutorial:

| Step | Tutorial |
|---|---|
| Define a time and frequency grid | 2 Time and Frequency Grids |
| Create an input pulse (the pump) | 3 Pulses |
| Define a waveguide: dispersion, poling, nonlinearity | 4 Waveguides |
| Propagate and analyse the output | 5 SHG |

## Installation

From the repository root, install snow and the extras used by the tutorials:

```
pip install -e ".[tutorials]"
```

The `-e` (editable) install means changes you pull from GitHub take effect without reinstalling. If you would rather not install anything, the tutorial notebooks also add the repository root to `sys.path`, so they run from the `tutorials` folder as they are.

The package needs Python 3.10 or newer, NumPy, SciPy and Matplotlib. The tutorials and the OPO module also use `colorcet`.

To check the installation, import snow and list the available materials:

In [1]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))  # use this repo's snow package

import snow.materials as materials

print(materials.available())

['Air', 'GaP', 'LN_MgO_e', 'LN_MgO_e_T', 'LN_MgO_o', 'LN_MgO_o_T', 'LN_e', 'LN_o', 'LT_MgO_e', 'LT_SLT', 'Sapphire', 'SiO2']


## References

1. M. Conforti, F. Baronio, and C. De Angelis, "Nonlinear envelope equation for broadband optical pulses in quadratic media," Phys. Rev. A **81**, 053841 (2010), doi: [10.1103/PhysRevA.81.053841](https://doi.org/10.1103/PhysRevA.81.053841). The first full description of the NEE.
2. T. Brabec and F. Krausz, "Nonlinear Optical Pulse Propagation in the Single-Cycle Regime," Phys. Rev. Lett. **78**, 3282 (1997), doi: [10.1103/PhysRevLett.78.3282](https://doi.org/10.1103/PhysRevLett.78.3282). The origin of the envelope approach for single-cycle pulses.

snow uses a frequency-domain formulation that differs from these papers in the details.